In [1]:
import pandas as pd
import numpy as np
from src.kmnri import kmnri

In [2]:
data = pd.read_csv("./data/data.csv")
data = data.loc[:, ["status", "time", "pred_base", "pred_big"]]
data

,status,time,pred_base,pred_big
0,1,306,0.433375,0.369498
1,1,455,0.288605,0.221848
2,0,1010,0.076872,0.043726
3,1,210,0.681076,0.626726
4,1,883,0.062970,0.035061
...,...,...,...,...
223,0,188,0.626521,0.577660
224,0,191,0.791519,0.746383
225,0,105,0.826945,0.875352
226,0,174,0.735391,0.693110


In [3]:
import numpy as np
from scipy.stats import norm

def categ_nri_boot(risklimits, pold, pnew, ftime, censvar, Time, Nboot):
    v = np.where(ftime > Time)[0]
    if len(v) > 0:
        censvar[v] = 0
        ftime[v] = Time
    
    Data = np.column_stack((pold, pnew, ftime, censvar))
    Data = Data[~np.isnan(Data).any(axis=1)]

    def kmnri_boot(data, indices):
        data = data[indices]
        a = kmnri(risklimits=risklimits,
                  pold=data[:, 0],
                  pnew=data[:, 1],
                  ftime=data[:, 2],
                  censvar=data[:, 3],
                  weight=None)
        nri_ev = a['casesup'] - a['casesdown']
        nri_nev = a['noncasesdown'] - a['noncasesup']
        nri = nri_ev + nri_nev
        return np.array([nri_ev, nri_nev, nri])

    # Calculate SE and CIs by bootstrapping
    Nboot = int(Nboot)
    results = []
    for _ in range(Nboot):
        indices = np.random.choice(Data.shape[0], size=Data.shape[0], replace=True)
        boot_result = kmnri_boot(Data, indices)
        results.append(boot_result)
    results_boot = np.array(results)
    SE = np.std(results_boot, axis=0)

    # Calculate NRI and NRI
    nri_result = kmnri(risklimits=risklimits,
                  pold=Data[:, 0],
                  pnew=Data[:, 1],
                  ftime=Data[:, 2],
                  censvar=Data[:, 3],
                  weight=None)
    print(nri_result)
    nri_ev = nri_result['casesup'] - nri_result['casesdown']
    nri_nev = nri_result['noncasesdown'] - nri_result['noncasesup']
    nri = nri = nri_ev + nri_nev
    
    out = {
        'nri': nri,
        'nri.SE': SE[2],
        'nri.p': 2 * norm.cdf(-np.abs(nri) / SE[2]),
        'nri.ev': nri_ev,
        'nri.ev.SE': SE[0],
        'nri.ev.p': 2 * norm.cdf(-np.abs(nri_ev) / SE[0]),
        'nri.nev': nri_nev,
        'nri.nev.SE': SE[1],
        'nri.nev.p': 2 * norm.cdf(-np.abs(nri_nev) / SE[1]),
        'N': Data.shape[0],
        'Nevent': np.sum(Data[:, 3] == 1)
    }
    
    return out

def process_nri(x):
    qqq = norm.ppf(0.975)
    
    out = {
        'nri': x['nri'],
        'nri.left': x['nri'] - qqq * x['nri.SE'],
        'nri.right': x['nri'] + qqq * x['nri.SE'],
        'nri.ev': x['nri.ev'],
        'nri.ev.left': x['nri.ev'] - qqq * x['nri.ev.SE'],
        'nri.ev.right': x['nri.ev'] + qqq * x['nri.ev.SE'],
        'nri.nev': x['nri.nev'],
        'nri.nev.left': x['nri.nev'] - qqq * x['nri.nev.SE'],
        'nri.nev.right': x['nri.nev'] + qqq * x['nri.nev.SE'],
        'N': x['N'],
        'Nevent': x['Nevent']
    }
    
    return pd.DataFrame(out)

def wrapper(pred):
    result = categ_nri_boot(risklimits=risklimits_nri_score,
                            pold=data[f"pred_{preds[0]}"],
                            pnew=data[f"pred_{pred}"],
                            ftime=data[event_time],
                            censvar=data[event],
                            Time=time_span,
                            Nboot=r_boot_nri)
    return result
    

In [4]:
preds = ["base", "big"]
event_time = "time"
event = "status"
time_span = 600
r_boot_nri = 50
risklimits_nri_score = [0.9]

In [5]:
nri_matrix_ac = [wrapper(pred) for pred in preds[1:]]
nri_matrix_ac = pd.DataFrame(nri_matrix_ac)

nri_matrix_ac = process_nri(nri_matrix_ac)

/var/folders/wk/cgsrf1m50qx268ccx7b33rdr0000gn/T/ipykernel_7856/1442390866.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  censvar[v] = 0
/var/folders/wk/cgsrf1m50qx268ccx7b33rdr0000gn/T/ipykernel_7856/1442390866.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ftime[v] = Time


{'casesup': 0.005577179362348333, 'casesdown': 0.027885896811741664, 'noncasesup': 0.020534768522466336, 'noncasesdown': 0.0}


In [6]:
nri_matrix_ac

,nri,nri.left,nri.right,nri.ev,nri.ev.left,nri.ev.right,nri.nev,nri.nev.left,nri.nev.right,N,Nevent
0,-0.042843,-0.089788,0.004101,-0.022309,-0.046172,0.001554,-0.020535,-0.062972,0.021902,228,148
